In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/blessingtosin/rental-info/rental_info.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/blessingtosin/rental-info/rental_info.csv')

In [3]:
df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   rental_date       15861 non-null  object 
 1   return_date       15861 non-null  object 
 2   amount            15861 non-null  float64
 3   release_year      15861 non-null  float64
 4   rental_rate       15861 non-null  float64
 5   length            15861 non-null  float64
 6   replacement_cost  15861 non-null  float64
 7   special_features  15861 non-null  object 
 8   NC-17             15861 non-null  int64  
 9   PG                15861 non-null  int64  
 10  PG-13             15861 non-null  int64  
 11  R                 15861 non-null  int64  
 12  amount_2          15861 non-null  float64
 13  length_2          15861 non-null  float64
 14  rental_rate_2     15861 non-null  float64
dtypes: float64(8), int64(4), object(3)
memory usage: 1.8+ MB


In [5]:
df.isnull().sum()

rental_date         0
return_date         0
amount              0
release_year        0
rental_rate         0
length              0
replacement_cost    0
special_features    0
NC-17               0
PG                  0
PG-13               0
R                   0
amount_2            0
length_2            0
rental_rate_2       0
dtype: int64

In [6]:
df['rental_date'] = pd.to_datetime(df['rental_date'])
df['return_date'] = pd.to_datetime(df['return_date'])

In [7]:
df['rental_length_days'] = (df['return_date'] - df['rental_date']).dt.days
df['rental_length_days'].head()

0    3
1    2
2    7
3    2
4    4
Name: rental_length_days, dtype: int64

In [8]:
df['deleted_scenes'] = df['special_features'].str.contains('Deleted Scenes')
df['behind_the_scenes'] = df['special_features'].str.contains('Behind the Scenes')

In [9]:
X = df.drop(columns=['return_date', 'rental_date', 'special_features', 'rental_length_days'])
y = df['rental_length_days']

In [10]:
X.head()

,amount,release_year,rental_rate,length,replacement_cost,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,deleted_scenes,behind_the_scenes
0,2.99,2005.0,2.99,126.0,16.99,0,0,0,1,8.9401,15876.0,8.9401,False,True
1,2.99,2005.0,2.99,126.0,16.99,0,0,0,1,8.9401,15876.0,8.9401,False,True
2,2.99,2005.0,2.99,126.0,16.99,0,0,0,1,8.9401,15876.0,8.9401,False,True
3,2.99,2005.0,2.99,126.0,16.99,0,0,0,1,8.9401,15876.0,8.9401,False,True
4,2.99,2005.0,2.99,126.0,16.99,0,0,0,1,8.9401,15876.0,8.9401,False,True


In [11]:
y.head()

0    3
1    2
2    7
3    2
4    4
Name: rental_length_days, dtype: int64

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=9)

In [13]:
X_test.head()

,amount,release_year,rental_rate,length,replacement_cost,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,deleted_scenes,behind_the_scenes
15067,4.99,2005.0,0.99,184.0,9.99,0,0,1,0,24.9001,33856.0,0.9801,True,True
3808,4.99,2005.0,4.99,179.0,29.99,0,0,0,1,24.9001,32041.0,24.9001,False,True
1015,4.99,2007.0,4.99,73.0,17.99,0,1,0,0,24.9001,5329.0,24.9001,True,True
12617,4.99,2009.0,0.99,172.0,14.99,0,0,0,1,24.9001,29584.0,0.9801,False,True
1711,4.99,2007.0,4.99,91.0,16.99,0,0,1,0,24.9001,8281.0,24.9001,True,True


In [14]:
linear_regression = LinearRegression() 
ridge_regression = Ridge(random_state=9) 
lasso_regression = Lasso(random_state=9)
random_forest = RandomForestRegressor(random_state=9)

In [15]:
linear_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', linear_regression)
])

In [16]:
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ridge_regression)
])

In [17]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', lasso_regression)
])

In [18]:
random_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', random_forest)
])

In [19]:
ridge_params = {
    'model__alpha': [0.1, 1.0, 5.0, 10.0]
}

lasso_params = {
    'model__alpha': [0.01, 0.05, 0.1, 0.5]
}

random_params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

In [20]:
models = {
    'Linear Regression': (linear_pipeline),
    'Ridge Regression': (ridge_pipeline, ridge_params),
    'Lasso Regression': (random_pipeline, random_params)
}

In [21]:
best_trained_models = {} 
results = {}

In [22]:
linear_pipeline.fit(X_train, y_train) 
y_pred_linear = linear_pipeline.predict(X_test)
mse_linear = mean_squared_error(y_test, y_pred_linear) 
best_trained_models['Linear Regression'] = linear_pipeline
results['Linear Regression'] = mse_linear
print(f"Linear MSE: {mse_linear:.4f}")

Linear MSE: 2.9417


In [23]:
ridge_grid = GridSearchCV(
    ridge_pipeline, ridge_params, cv=5,
    scoring='neg_mean_squared_error', n_jobs=-1
)
ridge_grid.fit(X_train, y_train) 
best_ridge = ridge_grid.best_estimator_
y_pred_ridge = best_ridge.predict(X_test) 
mse_ridge = mean_squared_error(y_test, y_pred_ridge) 

results['Ridge Regression'] = mse_ridge
best_trained_models['Ridge Regression'] = best_ridge
print(f"Ridge MSE: {mse_ridge:.4f}")

Ridge MSE: 2.9417


In [24]:
lasso_grid = GridSearchCV(
    lasso_pipeline, lasso_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)
lasso_grid.fit(X_train, y_train)
best_lasso = lasso_grid.best_estimator_ 
y_pred_lasso = best_lasso.predict(X_test) 
mse_lasso = mean_squared_error(y_test, y_pred_lasso) 

best_trained_models['Lasso Regression'] = best_lasso
results['Lasso Regression'] = mse_lasso 

print(f"Lasso: {mse_lasso:.4f}")

Lasso: 2.9505


In [25]:
random_grid = GridSearchCV(
    random_pipeline, random_params, cv=5, scoring='neg_mean_squared_error', n_jobs=-1
)

random_grid.fit(X_train, y_train) 
best_random = random_grid.best_estimator_ 
y_pred_random = best_random.predict(X_test) 
mse_random = mean_squared_error(y_test, y_pred_random) 

best_trained_models['Random Forest'] = best_model
results['Random Forest'] = mse_random
print(f"Random MSE: {mse_random:.4f}")

NameError: name 'best_model' is not defined

In [ ]:
best_model_name = min(results, key=results.get) 
best_model = best_trained_models[best_model_name] 
best_mse = results[best_model_name]

In [ ]:
print(f"BEST MODEL: {best_model_name}") 
print(f"MSE: {best_mse:.4f}") 
print(f"Requirement met the (MSE <= 3.0): {best_mse <= 3.0}")